In [ ]:
import pandas as pd

X_wo_test = pd.read_csv("X_wo_test.csv")
X_wo_train = pd.read_csv("X_wo_train.csv")
y_wo_test = pd.read_csv("y_wo_test.csv")
y_wo_train = pd.read_csv("y_wo_train.csv")



mask_train = (y_wo_train["price"] >= 500) & (y_wo_train["price"] <= 100000)

X_wo_train = X_wo_train.loc[mask_train]
y_wo_train = y_wo_train.loc[mask_train]

mask_test = (y_wo_test["price"] >= 500) & (y_wo_test["price"] <= 100000)

X_wo_test = X_wo_test.loc[mask_test]
y_wo_test = y_wo_test.loc[mask_test]

In [ ]:
from datetime import datetime

# Otteniamo l'anno corrente 
anno_corrente = datetime.now().year
print(anno_corrente)

# Creiamo la nuova feature "eta_auto" sottraendo l'anno di immatricolazione
X_wo_train['eta_auto'] = (anno_corrente - X_wo_train['year']).astype(int)
X_wo_test['eta_auto'] = (anno_corrente - X_wo_test['year']).astype(int)


# rimuovere la colonna originale
X_wo_train = X_wo_train.drop(columns=['year'])
X_wo_test = X_wo_test.drop(columns=['year'])


2026


In [26]:
categorical_cols = [
    "manufacturer", "model", "condition", "cylinders", "fuel",
    "title_status", "transmission", "drive", "size", "type",
    "paint_color", "state"
]

In [27]:
for col in categorical_cols:
    X_wo_train[col] = X_wo_train[col].astype("string")
    X_wo_test[col] = X_wo_test[col].astype("string")


In [ ]:
from catboost import CatBoostRegressor
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import optuna


# ottimizzazione
def objective(trial):

    params = {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "random_seed": 42,
        "verbose": 0,

        # iperparametri
        "iterations": trial.suggest_int("iterations", 300, 800),
        "depth": trial.suggest_int("depth", 4, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 20),
        "random_strength": trial.suggest_float("random_strength", 1, 30),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 1),
        "border_count": trial.suggest_int("border_count", 32, 255)
    }

    model = CatBoostRegressor(**params)

    model.fit(
        X_wo_train, y_wo_train,
        eval_set=(X_wo_test, y_wo_test),
        cat_features=categorical_cols,
        verbose=0
    )

    y_preds = model.predict(X_wo_test)
    #METRICHE
    mae = mean_absolute_error(y_wo_test, y_preds)
    mse = mean_squared_error(y_wo_test, y_preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_wo_test, y_preds)

    # Salviamo le metriche nel trial 
    trial.set_user_attr("mse", mse)
    trial.set_user_attr("rmse", rmse)
    trial.set_user_attr("r2", r2)

    return mae  


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Miglior MAE:", study.best_value)
print("Migliori parametri:", study.best_params)

best_trial = study.best_trial

print("MSE:", best_trial.user_attrs["mse"])
print("RMSE:", best_trial.user_attrs["rmse"])
print("R²:", best_trial.user_attrs["r2"])





c:\Users\diego\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-06-04 11:09:03,667] A new study created in memory with name: no-name-2d5db566-000b-454c-95da-f0ab29b51fbd
[I 2026-06-04 11:10:57,439] Trial 0 finished with value: 2868.475258207804 and parameters: {'iterations': 451, 'depth': 7, 'learning_rate': 0.16754832040155931, 'l2_leaf_reg': 19.6806500929752, 'random_strength': 26.466930233364288, 'bagging_temperature': 0.04902586149514654, 'border_count': 107}. Best is trial 0 with value: 2868.475258207804.
[I 2026-06-04 11:13:14,944] Trial 1 finished with value: 2943.2988438093457 and parameters: {'iterations': 547, 'depth': 7, 'learning_rate': 0.09094006279685413, 'l2_leaf_reg': 19.630132812939298, 'random_strength': 23.830981811974357, 'bagging_temperature': 0.5

Miglior MAE: 2588.772200052698
Migliori parametri: {'iterations': 765, 'depth': 8, 'learning_rate': 0.1750228835778352, 'l2_leaf_reg': 12.528239943993052, 'random_strength': 4.503171494760372, 'bagging_temperature': 0.08840320358574666, 'border_count': 150}
MSE: 17531285.94400452
RMSE: 4187.037848408409
R²: 0.8973006356952551
